In [29]:
import warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")           # non-interactive backend for file output
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

In [30]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import (
    train_test_split,
    cross_val_score,
    StratifiedKFold,
)
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    roc_curve,
    precision_recall_curve,
    average_precision_score,
    ConfusionMatrixDisplay,
)

warnings.filterwarnings("ignore")

In [31]:
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
#print("BANK MARKETING — RANDOM FOREST CLASSIFIER")

IMPORTING DATA

In [32]:
FILE_PATH = "/content/drive/MyDrive/RanfomForest_Banking/bank_full.xlsx"
df = pd.read_excel(FILE_PATH)

print(f"Loaded  : {df.shape[0]:,} rows x {df.shape[1]} columns")
print(f"Columns : {list(df.columns)}")

Loaded  : 45,211 rows x 17 columns
Columns : ['age', 'job', 'marital', 'education', 'default', 'balance', 'housing', 'loan', 'contact', 'day', 'month', 'duration', 'campaign', 'pdays', 'previous', 'poutcome', 'y']


EXPLORING DATA

In [33]:
print("Exploratory snapshot")

target_counts = df["y"].value_counts()
imbalance_ratio = target_counts["no"] / target_counts["yes"]
print(f"Target distribution: {target_counts.to_string()}")
print(f"Imbalance ratio   : {imbalance_ratio:.1f} : 1  (no : yes)")

print(f"Null values       : {df.isnull().sum().sum()} (explicit)")
print(f"'unknown' encoded : treated as a valid category; " f"keeps population representative.")

print("\nNumeric feature summary:")
print(df.describe().round(2).to_string())

Exploratory snapshot
Target distribution: y
no     39922
yes     5289
Imbalance ratio   : 7.5 : 1  (no : yes)
Null values       : 0 (explicit)
'unknown' encoded : treated as a valid category; keeps population representative.

Numeric feature summary:
            age    balance       day  duration  campaign     pdays  previous
count  45211.00   45211.00  45211.00  45211.00  45211.00  45211.00  45211.00
mean      40.94    1362.27     15.81    258.16      2.76     40.20      0.58
std       10.62    3044.77      8.32    257.53      3.10    100.13      2.30
min       18.00   -8019.00      1.00      0.00      1.00     -1.00      0.00
25%       33.00      72.00      8.00    103.00      1.00     -1.00      0.00
50%       39.00     448.00     16.00    180.00      2.00     -1.00      0.00
75%       48.00    1428.00     21.00    319.00      3.00     -1.00      0.00
max       95.00  102127.00     31.00   4918.00     63.00    871.00    275.00


DATA PREPROCESSING

In [34]:
df = df.drop(columns=["duration"])
print("Dropped 'duration' (post-call leakage variable)")

Dropped 'duration' (post-call leakage variable)


In [35]:
df["was_contacted_before"] = (df["pdays"] != -1).astype(int)
print("Engineered 'was_contacted_before' from pdays sentinel (-1)")

Engineered 'was_contacted_before' from pdays sentinel (-1)


In [36]:
df["y"] = (df["y"] == "yes").astype(int)
print(f"Target encoded: yes -> 1 ({df['y'].sum():,}), " f"no -> 0 ({(df['y']==0).sum():,})")

Target encoded: yes -> 1 (5,289), no -> 0 (39,922)


In [37]:
categorical_cols = [
    "job", "marital", "education", "contact", "month", "poutcome"
]
df = pd.get_dummies(df, columns=categorical_cols, drop_first=False)
print(f"One-hot encoded {len(categorical_cols)} categorical columns")

One-hot encoded 6 categorical columns


In [38]:
binary_cols = ["default", "housing", "loan"]
le = LabelEncoder()
for col in binary_cols:
    df[col] = le.fit_transform(df[col])
print(f"Label-encoded binary columns: {binary_cols}")

Label-encoded binary columns: ['default', 'housing', 'loan']


In [39]:
X = df.drop(columns=["y"])
y = df["y"]
feature_names = X.columns.tolist()
print(f"Final feature matrix : {X.shape[0]:,} rows × {X.shape[1]} cols")

Final feature matrix : 45,211 rows × 48 cols


In [40]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y,
)
print(f"Train : {X_train.shape[0]:,} rows  |  " f"Test  : {X_test.shape[0]:,} rows  (80/20 stratified)")

print(f"Train positives: {y_train.sum():,} " f"({y_train.mean()*100:.1f}%)  |  " f"Test positives: {y_test.sum():,} " f"({y_test.mean()*100:.1f}%)")

Train : 36,168 rows  |  Test  : 9,043 rows  (80/20 stratified)
Train positives: 4,231 (11.7%)  |  Test positives: 1,058 (11.7%)


HYPERPARAMETER TUNING

In [41]:
candidate_configs = [
    # Config A — balanced default: sqrt features, no depth cap
    dict(n_estimators=200, max_depth=None,  min_samples_split=5,  min_samples_leaf=2, max_features="sqrt"),

    # Config B — depth-capped: prevents overfit on noisy features
    dict(n_estimators=200, max_depth=20,    min_samples_split=5,  min_samples_leaf=2, max_features="sqrt"),

    # Config C — stronger regularisation: larger leaf + split thresholds
    dict(n_estimators=150, max_depth=None,  min_samples_split=10, min_samples_leaf=4, max_features="sqrt"),

    # Config D — log2 features: reduces correlation between trees further
    dict(n_estimators=150, max_depth=None,  min_samples_split=5,  min_samples_leaf=2, max_features="log2"),
]

In [42]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

best_cv_auc, best_params = 0.0, None
print(f"Evaluating {len(candidate_configs)} configs with 5-fold StratifiedCV")

Evaluating 4 configs with 5-fold StratifiedCV


In [43]:
for cfg in candidate_configs:
    rf_candidate = RandomForestClassifier(
        **cfg,
        class_weight="balanced",  # handle 88/12 imbalance
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )
    scores = cross_val_score(rf_candidate, X_train, y_train,
                             cv=cv, scoring="roc_auc", n_jobs=-1)
    mean_s = scores.mean()
    print(f"         AUC={mean_s:.4f} (±{scores.std():.4f})  |  {cfg}")
    if mean_s > best_cv_auc:
        best_cv_auc, best_params = mean_s, cfg

print(f"\n       Best CV AUC-ROC : {best_cv_auc:.4f}")
print(f"       Best params    : {best_params}")

         AUC=0.7916 (±0.0039)  |  {'n_estimators': 200, 'max_depth': None, 'min_samples_split': 5, 'min_samples_leaf': 2, 'max_features': 'sqrt'}
         AUC=0.7920 (±0.0029)  |  {'n_estimators': 200, 'max_depth': 20, 'min_samples_split': 5, 'min_samples_leaf': 2, 'max_features': 'sqrt'}
         AUC=0.7934 (±0.0041)  |  {'n_estimators': 150, 'max_depth': None, 'min_samples_split': 10, 'min_samples_leaf': 4, 'max_features': 'sqrt'}
         AUC=0.7901 (±0.0036)  |  {'n_estimators': 150, 'max_depth': None, 'min_samples_split': 5, 'min_samples_leaf': 2, 'max_features': 'log2'}

       Best CV AUC-ROC : 0.7934
       Best params    : {'n_estimators': 150, 'max_depth': None, 'min_samples_split': 10, 'min_samples_leaf': 4, 'max_features': 'sqrt'}


FINAL MODEL

In [44]:
best_rf = RandomForestClassifier(
    **best_params,
    class_weight="balanced",
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
best_rf.fit(X_train, y_train)

RandomForestClassifier(class_weight='balanced', min_samples_leaf=4,
                       min_samples_split=10, n_estimators=150, n_jobs=-1,
                       random_state=42)

In [45]:
y_pred = best_rf.predict(X_test)
y_prob = best_rf.predict_proba(X_test)[:, 1]

EVALUATION

In [46]:
print("  CLASSIFICATION REPORT")

report = classification_report(y_test, y_pred, target_names=["No", "Yes"])
print(report)

  CLASSIFICATION REPORT
              precision    recall  f1-score   support

          No       0.94      0.92      0.93      7985
         Yes       0.45      0.52      0.49      1058

    accuracy                           0.87      9043
   macro avg       0.69      0.72      0.71      9043
weighted avg       0.88      0.87      0.87      9043



In [47]:
auc_roc = roc_auc_score(y_test, y_prob)
avg_precision = average_precision_score(y_test, y_prob)
print(f"  AUC-ROC             : {auc_roc:.4f}")
print(f"  Avg Precision (PR)  : {avg_precision:.4f}")


  AUC-ROC             : 0.8045
  Avg Precision (PR)  : 0.4543


In [48]:
fig = plt.figure(figsize=(22, 18))
fig.patch.set_facecolor("#F7F9FC")
gs = gridspec.GridSpec(3, 3, figure=fig, hspace=0.45, wspace=0.38)

PALETTE   = {"yes": "#2196F3", "no": "#EF5350"}
BLUE      = "#2196F3"
RED       = "#EF5350"
DARK      = "#1A237E"
ACCENT    = "#0D47A1"
LIGHTGRAY = "#ECEFF1"

# ── Helper: axis styling ────────────────────────────────────────────────────
def style_ax(ax, title, xlabel="", ylabel=""):
    ax.set_facecolor(LIGHTGRAY)
    ax.set_title(title, fontsize=13, fontweight="bold", color=DARK, pad=10)
    ax.set_xlabel(xlabel, fontsize=10, color="#37474F")
    ax.set_ylabel(ylabel, fontsize=10, color="#37474F")
    ax.tick_params(colors="#37474F", labelsize=9)
    for spine in ax.spines.values():
        spine.set_edgecolor("#B0BEC5")
    ax.grid(axis="y", color="white", linewidth=0.8)


# ─── Panel 1: Target Class Distribution ───────────────────────────────────
ax1 = fig.add_subplot(gs[0, 0])
counts   = y.value_counts()
bars     = ax1.bar(
    ["No (0)", "Yes (1)"],
    counts.values,
    color=[RED, BLUE],
    width=0.5,
    edgecolor="white",
    linewidth=1.2,
)
for bar, val in zip(bars, counts.values):
    ax1.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 400,
        f"{val:,}\n({val/len(y)*100:.1f}%)",
        ha="center", va="bottom", fontsize=9, fontweight="bold", color=DARK,
    )
style_ax(ax1, "Target Class Distribution", "Class", "Count")
ax1.set_ylim(0, counts.max() * 1.18)


# ─── Panel 2: Confusion Matrix ─────────────────────────────────────────────
ax2 = fig.add_subplot(gs[0, 1])
cm  = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["No", "Yes"])
disp.plot(ax=ax2, colorbar=False, cmap="Blues")
ax2.set_title("Confusion Matrix", fontsize=13, fontweight="bold",
              color=DARK, pad=10)
ax2.set_facecolor(LIGHTGRAY)
# Annotate TN/FP/FN/TP labels
tn, fp, fn, tp = cm.ravel()
ax2.text(0, 0.38, f"TN\n{tn:,}", ha="center", va="center",
         fontsize=8, color="navy", alpha=0.6)
ax2.text(1, 0.38, f"FP\n{fp:,}", ha="center", va="center",
         fontsize=8, color="darkred", alpha=0.6)
ax2.text(0, 1.38, f"FN\n{fn:,}", ha="center", va="center",
         fontsize=8, color="darkred", alpha=0.6)
ax2.text(1, 1.38, f"TP\n{tp:,}", ha="center", va="center",
         fontsize=8, color="darkgreen", alpha=0.6)


# ─── Panel 3: ROC Curve ────────────────────────────────────────────────────
ax3 = fig.add_subplot(gs[0, 2])
fpr, tpr, _ = roc_curve(y_test, y_prob)
ax3.plot(fpr, tpr, color=BLUE, lw=2.5, label=f"RF  (AUC = {auc_roc:.3f})")
ax3.plot([0, 1], [0, 1], "k--", lw=1.2, alpha=0.4, label="Random (AUC = 0.5)")
ax3.fill_between(fpr, tpr, alpha=0.08, color=BLUE)
ax3.legend(fontsize=9, loc="lower right")
style_ax(ax3, "ROC Curve", "False Positive Rate", "True Positive Rate")
ax3.set_xlim([-0.01, 1.01]); ax3.set_ylim([-0.01, 1.05])


# ─── Panel 4: Precision-Recall Curve ──────────────────────────────────────
ax4 = fig.add_subplot(gs[1, 0])
precision_vals, recall_vals, _ = precision_recall_curve(y_test, y_prob)
ax4.plot(recall_vals, precision_vals, color=RED, lw=2.5,
         label=f"RF  (AP = {avg_precision:.3f})")
baseline = y_test.mean()
ax4.axhline(baseline, color="gray", linestyle="--", lw=1.2,
            label=f"Baseline ({baseline:.2f})")
ax4.fill_between(recall_vals, precision_vals, alpha=0.08, color=RED)
ax4.legend(fontsize=9)
style_ax(ax4, "Precision-Recall Curve", "Recall", "Precision")
ax4.set_xlim([-0.01, 1.01]); ax4.set_ylim([0, 1.05])


# ─── Panel 5: Feature Importance (Top 20) ─────────────────────────────────
ax5 = fig.add_subplot(gs[1, 1:])
importances = best_rf.feature_importances_
fi_df       = pd.DataFrame({
    "feature"   : feature_names,
    "importance": importances,
}).sort_values("importance", ascending=False).head(20)

colors = [BLUE if i < 5 else "#90CAF9" for i in range(len(fi_df))]
bars5  = ax5.barh(
    fi_df["feature"][::-1],
    fi_df["importance"][::-1],
    color=colors[::-1],
    edgecolor="white",
    linewidth=0.6,
)
for bar in bars5:
    w = bar.get_width()
    ax5.text(w + 0.001, bar.get_y() + bar.get_height() / 2,
             f"{w:.3f}", va="center", fontsize=8, color=DARK)
style_ax(ax5, "Top 20 Feature Importances (Mean Decrease in Impurity)",
         "Importance Score", "")
ax5.set_facecolor(LIGHTGRAY)
ax5.grid(axis="x", color="white", linewidth=0.8)
ax5.grid(axis="y", visible=False)


# ─── Panel 6: Probability Distribution by True Class ──────────────────────
ax6 = fig.add_subplot(gs[2, 0])
prob_no  = y_prob[y_test == 0]
prob_yes = y_prob[y_test == 1]
ax6.hist(prob_no,  bins=40, alpha=0.65, color=RED,  label="Actual No",
         density=True, edgecolor="white", linewidth=0.4)
ax6.hist(prob_yes, bins=40, alpha=0.65, color=BLUE, label="Actual Yes",
         density=True, edgecolor="white", linewidth=0.4)
ax6.axvline(0.5, color="black", linestyle="--", lw=1.5, label="Default threshold (0.5)")
ax6.legend(fontsize=9)
style_ax(ax6, "Predicted Probability Distribution",
         "P(Subscription)", "Density")


# ─── Panel 7: Precision / Recall / F1 at varying thresholds ───────────────
ax7 = fig.add_subplot(gs[2, 1])
thresholds = np.linspace(0.05, 0.95, 100)
prec_list, rec_list, f1_list = [], [], []
for t in thresholds:
    y_t = (y_prob >= t).astype(int)
    tp_ = ((y_t == 1) & (y_test == 1)).sum()
    fp_ = ((y_t == 1) & (y_test == 0)).sum()
    fn_ = ((y_t == 0) & (y_test == 1)).sum()
    p   = tp_ / (tp_ + fp_) if (tp_ + fp_) > 0 else 0
    r   = tp_ / (tp_ + fn_) if (tp_ + fn_) > 0 else 0
    f1  = 2*p*r/(p+r)       if (p+r)       > 0 else 0
    prec_list.append(p); rec_list.append(r); f1_list.append(f1)

ax7.plot(thresholds, prec_list, color=RED,  lw=2, label="Precision")
ax7.plot(thresholds, rec_list,  color=BLUE, lw=2, label="Recall")
ax7.plot(thresholds, f1_list,   color="#4CAF50", lw=2, label="F1")
ax7.axvline(0.5, color="black", linestyle="--", lw=1.2, alpha=0.5)
best_f1_idx = np.argmax(f1_list)
ax7.axvline(thresholds[best_f1_idx], color="#4CAF50",
            linestyle=":", lw=1.5,
            label=f"Best F1 @ {thresholds[best_f1_idx]:.2f}")
ax7.legend(fontsize=8)
style_ax(ax7, "Precision / Recall / F1 vs. Threshold",
         "Decision Threshold", "Score")
ax7.set_xlim([0, 1]); ax7.set_ylim([0, 1.05])


# ─── Panel 8: Customer Ranking — Cumulative Gains ─────────────────────────
ax8 = fig.add_subplot(gs[2, 2])
sorted_idx = np.argsort(y_prob)[::-1]
total_pos  = y_test.sum()
cum_pos    = np.cumsum(y_test.values[sorted_idx])
pct_pop    = np.arange(1, len(y_test) + 1) / len(y_test)
pct_pos    = cum_pos / total_pos

ax8.plot(pct_pop, pct_pos, color=BLUE, lw=2.5, label="RF Model")
ax8.plot([0, 1], [0, 1], "k--", lw=1.2, alpha=0.4, label="Random")
ax8.plot([0, total_pos / len(y_test), 1], [0, 1, 1],
         color="#4CAF50", lw=1.5, linestyle=":", label="Perfect")
# Annotate: contacting top 30% of ranked list
idx_30 = np.searchsorted(pct_pop, 0.30)
ax8.annotate(
    f"Top 30% → {pct_pos[idx_30]*100:.0f}% of subs",
    xy=(pct_pop[idx_30], pct_pos[idx_30]),
    xytext=(0.45, pct_pos[idx_30] - 0.12),
    arrowprops=dict(arrowstyle="->", color=DARK),
    fontsize=8.5, color=DARK,
)
ax8.legend(fontsize=9)
style_ax(ax8, "Cumulative Gains Chart (Customer Ranking)",
         "% Population Contacted", "% Subscriptions Captured")
ax8.set_xlim([0, 1]); ax8.set_ylim([0, 1.05])


# ── Master title ────────────────────────────────────────────────────────────
fig.suptitle(
    "Bank Marketing — Random Forest Model Report",
    fontsize=18, fontweight="bold", color=DARK, y=0.98,
)

OUTPUT_FIG = r"C:\Users\Ishayu\Downloads\bank_rf_model_report.png"
plt.savefig(OUTPUT_FIG, dpi=160, bbox_inches="tight",
            facecolor=fig.get_facecolor())
plt.close()
print(f"  Visualisation saved → {OUTPUT_FIG}")

  Visualisation saved → C:\Users\Ishayu\Downloads\bank_rf_model_report.png


CUSTOMER PROBABILITY RANKING TABLE (Business Output)

In [49]:
ranking_df = pd.DataFrame({
    "CustomerIndex"     : X_test.index,
    "P(Subscribe)"      : y_prob.round(4),
    "Predicted_Label"   : y_pred,
    "Actual_Label"      : y_test.values,
}).sort_values("P(Subscribe)", ascending=False).reset_index(drop=True)

ranking_df.index += 1   # 1-based rank
ranking_df["Rank"] = ranking_df.index

print(ranking_df[["Rank", "CustomerIndex", "P(Subscribe)",
                   "Predicted_Label", "Actual_Label"]].head(20).to_string())

print("\n  Interpretation:")
print("  → Sort customers by P(Subscribe) descending before each campaign.")
print("  → Contact only those above your chosen probability threshold")
print("    to maximise ROI and minimise wasted call spend.")


    Rank  CustomerIndex  P(Subscribe)  Predicted_Label  Actual_Label
1      1          43005        0.9700                1             1
2      2          43135        0.9660                1             0
3      3          43890        0.9647                1             1
4      4          43033        0.9615                1             1
5      5          42986        0.9577                1             1
6      6          43122        0.9569                1             1
7      7          42993        0.9533                1             1
8      8          43938        0.9514                1             1
9      9          45012        0.9514                1             1
10    10          41241        0.9478                1             1
11    11          43892        0.9473                1             1
12    12          43336        0.9455                1             1
13    13          42966        0.9455                1             1
14    14          42988        0.9

FINAL SCORECARD



In [50]:
print("  FINAL MODEL SCORECARD")
print("=" * 60)
best_f1_thresh = thresholds[np.argmax(f1_list)]

from sklearn.metrics import precision_score, recall_score, f1_score
y_pred_opt = (y_prob >= best_f1_thresh).astype(int)

print(f"  Best params (tuned)  : {best_params}")
print(f"  CV AUC-ROC (5-fold)  : {best_cv_auc:.4f}")
print(f"  Test AUC-ROC         : {auc_roc:.4f}")
print(f"  Avg Precision (PR)   : {avg_precision:.4f}")
print(f"  Default threshold (0.5):")
print(f"    Precision (Yes)    : "
      f"{precision_score(y_test, y_pred, pos_label=1):.4f}")
print(f"    Recall    (Yes)    : "
      f"{recall_score(y_test, y_pred, pos_label=1):.4f}")
print(f"    F1        (Yes)    : "
      f"{f1_score(y_test, y_pred, pos_label=1):.4f}")
print(f"  Optimal threshold ({best_f1_thresh:.2f}):")
print(f"    Precision (Yes)    : "
      f"{precision_score(y_test, y_pred_opt, pos_label=1):.4f}")
print(f"    Recall    (Yes)    : "
      f"{recall_score(y_test, y_pred_opt, pos_label=1):.4f}")
print(f"    F1        (Yes)    : "
      f"{f1_score(y_test, y_pred_opt, pos_label=1):.4f}")

  FINAL MODEL SCORECARD
  Best params (tuned)  : {'n_estimators': 150, 'max_depth': None, 'min_samples_split': 10, 'min_samples_leaf': 4, 'max_features': 'sqrt'}
  CV AUC-ROC (5-fold)  : 0.7934
  Test AUC-ROC         : 0.8045
  Avg Precision (PR)   : 0.4543
  Default threshold (0.5):
    Precision (Yes)    : 0.4531
    Recall    (Yes)    : 0.5246
    F1        (Yes)    : 0.4862
  Optimal threshold (0.51):
    Precision (Yes)    : 0.4636
    Recall    (Yes)    : 0.5123
    F1        (Yes)    : 0.4868
